# Query Translation
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some way as to improve retrival.

Semantic search on embeddings is hard to get right. Embedding long documents is especially challenging. User queries are a challenge too. If the user provides an ambigious query, they'll end up get an ambiguous matches from embeddings and consequently an ambguous answer. The ambiguous matches land up in the LLM's context from which comes the LLM's response, which could lead to hallucinations.

In this notebook, we'll discuss the following techniques, including what each technique does and when to use them:
1. Multi Query
2. RAG Fusion
3. Query Decomposition
4. Step-back Prompting
5. HYDE (**HY**pothetical **D**ocument **E**mbedding)

![Query Translation](../images/rag_query_translation.png)

## Multi-Query

**What is does**

* Takes the **user query** and asks the LLM to **generate several alternative** versions of that query.
* The goal is to **capture synonyms, different phrasings, and other angles** of the question.
* _Each expandeod query_ is sent to the vector store → all results are merged → RAG runs on combined context.
* The intuition is that by asking the LLM the same question in N different ways, we will get more relevents chunks of data into the context, thereby improving overall response.

**Why use it?**

Different phrasings capture different embeddings → retrieve more relevant chunks. Helps reduce “embedding mismatch” (e.g., synonyms, domain-specific terms).

**Example:**

User asks: _"How do I cook pasta quickly?"_
LLM generates (3 variations in this case):
* "fast ways to prepare pasta"
* "quick pasta cooking methods"
* "rapid spaghetti preparation"

All run → retrieve docs covering microwaving, pressure cooker, etc. (there could be duplicates, so generate a unique list of retrievals). The retriever might otherwise miss some if only the original query was used.

**Key point:** Multi-query **improves recall** by broadening query formulations.

**Subtle difference**
* Multi-Query **does not** break the question into sub-questions.
* It simply **generates alternative rewrites** of the same question.

The diagram below illustrates this technique.

![Multi Query](../images/multi_query.png)

In [14]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

In [15]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [16]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
# No longer using Google Embeddings as I have apparently exhausted my free quota :(
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/text-embedding-004", task_type="retrieval_document"
# )
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_rag_weng"

In [17]:
def create_or_load_embeddings(embeddings, faiss_store, chunk_size=300, chunk_overlap=50):
    """creates (if not available) or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        # text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        #     chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        # )
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [18]:
retriever = create_or_load_embeddings(embeddings, faiss_store)

Loading document from URL https://lilianweng.github.io/posts/2023-06-23-agent/. Please wait...
Loaded 1 documents from URL
Metadata of first document: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
First 200 chars of first document: 

      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a 
Chunking the PDF. Please wait...
Created 214 chunks
Creating embeddings. Please wait...
Local embeddings created at /home/mjbhobe/code/git-projects/learning_langchain/src/langchain_tutorial/Advanced RAG/../faiss_index_rag_weng


In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# from langchain_google_genai import ChatGoogleGenerativeAI

# Multi Query: Different formulations of same query
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. 

Original question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(template)

In [22]:
generate_queries = (
    prompt_perspectives | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke({"question": "What is task decomposition for LLM agents?"})

['Define task decomposition for large language model agents.',
 'Why do LLM agents employ task decomposition?',
 'How is task decomposition implemented or performed by LLM agents?',
 'What are the advantages of using task decomposition in LLM agent systems?',
 'Describe the process of breaking down complex problems for AI agents utilizing large language models.']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

In [23]:
from langchain.load import dumps, loads


def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs"""
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

In [24]:
# Retrieve
question = "What is task decomposition for LLM agents?"
# here we are firing multiple queries against the vector store, getting all the
# responses & creating a unique set from all the responses.
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question": question})
print(f"Got {len(docs)} documents")
for i, doc in enumerate(docs):
    console.print(
        Markdown(f"### Document {i+1}\n{doc.page_content[:50] + "..."}\n---\n")
    )

# and print the retrival chain too
console.print(f"Retrieval chain: {retrieval_chain}")

Got 8 documents
                                                       Document 1                                                       


                                 Reliability of natural language interface: Current...                                  
                                                       Document 2                                                       


                                 Building agents with LLM (large language model) as...                                  
                                                       Document 3                                                       


                                 Boiko et al. (2023) also looked into LLM-empowered...                                  
                                                       Document 4                                                       


                                 The system comprises of 4 stages: (1) Task plannin...                                  
        

In [25]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

multi_query_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = multi_query_rag_chain.invoke({"question": question})
console.print(f"[blue]Question: {question}[/blue]\n")
console.print(f"[yellow]AI Response:[/yellow]")
console.print(Markdown(response))

Question: What is task decomposition for LLM agents?

AI Response:
Task decomposition for LLM agents is the process where the agent breaks down large, complex tasks into smaller, more manageable subgoals. This enables efficient handling of complex tasks.

It can be achieved in several ways:

 1 By the LLM itself with simple prompting, such as "Steps for XYZ." or "What are the subgoals for achieving XYZ?".     
 2 By using task-specific instructions, for example, "Write a story outline." for writing a novel.                      
 3 With human inputs.                                                                                                   

In the context of task planning, the LLM functions as the brain, parsing user requests into multiple tasks, each with attributes like task type, ID, dependencies, and arguments, often guided by few-shot examples.


## RAG Fusion
(Think of it as **Multi-Query + ranking / scoring**)

**What is it?**

A **retrieval re-ranking technique** inspired by “Reciprocal Rank Fusion” (RRF) in information retrieval. 
* Generates multiple alternative queries → retrieves documents (multiple retrievals) → uses an algorithm (e.g., Reciprocal Rank Fusion) to rank documents more intelligently.
* You then **fuse/combine the rankes lists/results** from all queries using statistical fusion into one final ranking, _not naive concatenation_ (as you did in Multi-query).

**Why it’s better than Multi-Query**
* **Multi-Query**: retrieve from multiple queries → concatenate.
* **RAG Fusion**: =retrieve → rank → select best documents.

**Key idea**
* **Multiple phrasings + smart ranking → high recall AND high precision**.

**Subtle difference with Multi-Query**
* Both do query expansion, 
* But **RAG Fusion adds mathematical ranking, avoiding irrelevant noise**.

**How it works:**
* Each retrieval returns a ranked list (doc A rank=1, doc B rank=2, etc).
* Fusion scores docs by combining their **reciprocal ranks**:

$$
score(d) = \sum_{retrievers} \frac{1}{k+rank(d)}
$$
(with `k`= smoothening constant)
* Documents that appear across multiple queries/retrievers rise to the top.
* Reduces noise because only documents consistently relevant get boosted.

**Key Point:** RAG Fusion improves precision and robustness by rewarding cross-query/retriever consensus.

The diagram below illustrates this technique.

![Multi Query](../images/rag_fusion.png)

Below is the code to implement **RAG Fusion**. There will be a lot of duplicate code cells - this has been done intentionally to enable you to run 2 sections independently!

In [26]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

In [27]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [28]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
# No longer using Google Embeddings as I have apparently exhausted my free quota :(
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/text-embedding-004", task_type="retrieval_document"
# )
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_rag_weng"

In [29]:
def create_or_load_embeddings(embeddings, faiss_store, chunk_size=300, chunk_overlap=50):
    """creates (if not available) or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        # text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        #     chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        # )
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [30]:
retriever = create_or_load_embeddings(embeddings, faiss_store)

Loading existing embeddings from /home/mjbhobe/code/git-projects/learning_langchain/src/langchain_tutorial/Advanced RAG/../faiss_index_rag_weng


In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# from langchain_google_genai import ChatGoogleGenerativeAI

# RAG-Fusion: prompt
template = """You are a helpful assistant that generates multiple search queries 
based on a single input query.\n 
Return just a simple list of queries with no additional markup or text\n
Generate multiple search queries related to: {question} \n
Output ({num_queries} queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [33]:
generate_queries = (
    prompt_rag_fusion | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke(
    {
        "num_queries": 5,
        "question": "What is task decomposition for LLM agents?",
    }
)

['Task decomposition for LLM agents explained',
 'How LLM agents use task decomposition',
 'Techniques for task decomposition in large language models',
 'Benefits of task decomposition for LLM performance',
 'LLM agent architecture task breakdown']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

In [36]:
from langchain.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    """Reciprocal_rank_fusion that takes multiple lists of ranked documents
    and an optional parameter k used in the RRF formula"""

    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key 
            # (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

In [37]:
# Retrieve
from pydoc import doc

question = "What is task decomposition for LLM agents?"
# here we are firing multiple queries against the vector store, getting all the
# responses & creating a unique set from all the responses.
retrieval_chain_rf = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rf.invoke({"question": question, "num_queries": 5})
print(f"Got {len(docs)} documents")
# for i in range(5):
#     print(f"{docs[i]}\n\n")

# Note: docs -> List[(document, score)] (i.e. a list of tuples of (Document, score))
for i, (doc, score) in enumerate(docs[: len(docs) // 2]):
    print(f"Document #{i+1}\nContent: {doc.page_content}\nScore: {score}")

# # and print the retrival chain too
# console.print(f"Retrieval chain: {retrieval_chain}")

Got 9 documents
Document #1
Content: The system comprises of 4 stages:
(1) Task planning: LLM works as the brain and parses the user requests into multiple tasks. There are four attributes associated with each task: task type, ID, dependencies, and arguments. They use few-shot examples to guide LLM to do task parsing and planning.
Score: 0.0819672131147541
Document #2
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Score: 0.06666666666666667
Document #3
Content: Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it
Score: 0.04866871479

In [38]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

final_rag_chain_rag_fusion = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = final_rag_chain_rag_fusion.invoke({"question": question, "num_queries": 5})
console.print(Markdown(response))

Task decomposition for LLM agents is the process where the agent breaks down large, complex tasks into smaller, more manageable subgoals. This enables the efficient handling of complex tasks.

LLMs can perform task decomposition in several ways:

 • Simple prompting: Using prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?".                  
 • Task-specific instructions: Providing specific instructions, such as "Write a story outline" for a novel-writing task
 • Human inputs: Incorporating direct input from a human.                                                               

In this role, the LLM acts as the agent's brain, parsing user requests into these multiple, smaller tasks.


## Query Decomposition

**What is It?**
**Query decomposition** is a technique of breaking one complex user query into **several** _simpler_, _focused_ sub-queries; retrieving for each, and then combining the results. It’s usually done with an LLM:

1. **Input:** a long or multi-part user question.
2. **Decompose:** use an LLM prompt such as
    “Decompose this question into a list of simpler search queries.”
3. **Retrieve:** run each sub-query against your retriever/vector DB.
4. **Synthesize:** feed the retrieved chunks back into the LLM to build the final answer.

**Example**

User asks:

`Compare net profit trends and regulatory risks of Tesla over last 3 years.`

Decomposition may generate:
* "What are the net profit trends of Tesla in last 3 years?"
* "What are the regulatory risks faced by Tesla?"
* "How to compare these two aspects?"

**Key idea**
Break big task → smaller tasks → retrieve → combine.

**Subtle difference**
* **Multi-Query**: _same_ question phrased differently
* **Decomposition**: _different sub-questions_ covering _different aspects_.

**Why / When to Use Query Decomposition**

✅ Use it when:
* **Complex / multi-aspect questions:** 
    e.g., “Compare AutoGPT and BabyAGI, and explain how planning differs from memory.”

* **Broad tasks spanning sub-topics:**
    e.g., “Give me the pros/cons of hybrid search and explain when to use reciprocal rank fusion.”

* **Long, natural language queries:** with multiple clauses joined by “and”, “or”, “how … and also …”.

* **Poor retrieval recall:** when a single embedding search often misses pieces of the question.

🚫 Less helpful when:
* The query is **short and atomic** (e.g., “What is RAG Fusion?”).
* The corpus is tiny or each document already covers the entire topic.

Once the query is broken decomposed into individual queries, there are two broad techniques to retrieve responses:
* Answer individually
* Answer recursively

We'll cover decomposition & the two answering techniques in this section. You'll notice lot of repeating code to allow you to run this section independent of other sections.



In [21]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# I seem to have permanently exhausted rate limt on Google embeddings on free tier,
# I don't want to enable billing, so am switching to Cohere embeddings
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

In [22]:
# load API keys from .env files
load_dotenv(override=True)

# for colorful text output
# when using Console in a Notebook editor, especially in VS code, use the following
# alternate instantiation, rather than the default console = Console()

# console = Console()
console = Console(force_jupyter=False, force_terminal=True, width=120, soft_wrap=True)

In [23]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
embeddings = CohereEmbeddings(model="embed-english-v3.0")
faiss_store = pathlib.Path(os.getcwd()) / "../faiss_index_rag_qd"

In [24]:
def create_or_load_embeddings():
    """creates (if not available) or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        # embeddings = GoogleGenerativeAIEmbeddings(
        #     model="models/text-embedding-004",
        #     task_type="retrieval_document",
        # )
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        # embeddings = GoogleGenerativeAIEmbeddings(
        #     model="models/text-embedding-004",
        #     task_type="retrieval_document",
        # )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [25]:
retriever = create_or_load_embeddings()

Loading existing embeddings from C:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\Advanced RAG\..\faiss_index_rag_qd


As a first step, let us ask the LLM our question & check it's response without any query decomposition.

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [27]:
QUESTION = "What is task decomposition for LLM agents?"

In [28]:
template = ChatPromptTemplate.from_template(
    "Answer the following question:\n\n{question}"
)
simple_chain = template | llm | StrOutputParser()
response = simple_chain.invoke({"question": QUESTION})
console.print(Markdown(response))

Task decomposition for LLM agents is the process of breaking down a complex, high-level task into smaller, more manageable, and often sequential sub-tasks or steps. Each sub-task is simpler for the Large Language Model (LLM) to understand, process, and execute accurately, leading to a more reliable and robust overall performance.

                                  Why is Task Decomposition Necessary for LLM Agents?                                   

LLMs, despite their impressive capabilities, have limitations, especially when faced with multi-step reasoning, long contexts, or intricate logic within a single prompt. Task decomposition addresses these limitations by:

 1 Reducing Complexity: A single, complex prompt can overwhelm an LLM, leading to errors, hallucinations, or incomplete 
 2 Improving Accuracy and Reliability: By focusing on one clear sub-task at a time, the LLM is less likely to get sidetr
 3 Facilitating Multi-Step Reasoning: Many real-world problems require a sequence

Now let's ask LLM to decompose our query as we discussed above.

In [29]:
# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related 
to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that 
can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Generate just the list of queries. Don't generate any other text, such as numbering or additional quotes around the queries or markdown text \n
Output ({num_queries} queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

In [30]:
queries_generator = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
questions = queries_generator.invoke(
    {
        "num_queries": 5,
        "question": QUESTION,
    }
)
console.print(questions)

[
    'LLM agent task decomposition definition',
    'Why is task decomposition important for LLM agents',
    'Methods for task decomposition in large language model agents',
    'Benefits of task decomposition for LLM-powered agents',
    'Examples of task decomposition strategies for LLM agents'
]


### Answering Techniques for Query Decomposition
So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

Once we have the decomposed questions, we'll tweak the way the LLM responds to these questions.

#### Answering Individually
**How it works**:
* Break the big question into smaller sub-questions. 
* Retrieve documents **for each sub-question separately**. 
* Answer each sub-question independently using RAG.
* **Combine all sub-answers** into one final answer.

**Example**
_User asks_:

`Compare AI regulations in the US, EU, and China.`

_Possible Sub-questions generated_:

* "What are AI regulations in the US?"
* "What are AI regulations in the EU?"
* "What are AI regulations in China?"

_Process_:
* Retrieve for (1), answer (1)
* Retrieve for (2), answer (2)
* Retrieve for (3), answer (3)
* Then combine.

**Key Properties**
* Independent answers → high recall
* More context diversity
* Less chance of missing a sub-topic
* More expensive → many retrieval + LLM calls

**Subtle difference with _Anwer Recursively_** (which we will cover later)
* This method **does not use _intermediate answers_** to inform later ones.
* Each _sub-question is handled as if it is unrelated to the others_.

The image below illustrates this process visually:

![Multi Query](../images/ans_individually.png)

In [31]:
rag_prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. 
    Use three sentences maximum and keep the answer concise.\n
    Question: {question} \n
    Context: {context} \n
    Answer:"""
)

In [32]:
def retrieve_individual_qna(
    question, prompt_rag, sub_question_generator_chain, num_queries=5
):
    # Use our decomposition /
    sub_questions = sub_question_generator_chain.invoke(
        {"question": question, "num_queries": num_queries}
    )

    # Initialize a list to hold RAG chain results
    rag_results = []

    for sub_question in sub_questions:
        # Retrieve documents for each sub-question
        retrieved_docs = retriever.get_relevant_documents(sub_question)
        # Use retrieved documents and sub-question in RAG chain
        answer = (prompt_rag | llm | StrOutputParser()).invoke(
            {"context": retrieved_docs, "question": sub_question}
        )
        rag_results.append(answer)

    return sub_questions, rag_results


def format_qna_pairs(questions, answers):
    """Format Q and A pairs"""

    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()


In [33]:
questions, answers = retrieve_individual_qna(
    QUESTION, rag_prompt, queries_generator
)

C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_12972\3651332185.py:14: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(sub_question)


In [34]:
qna_pairs = format_qna_pairs(questions, answers)
print(qna_pairs)

Question 1: LLM agent task decomposition definition
Answer 1: LLM agent task decomposition is the process where an agent breaks down large, complex tasks into smaller, more manageable subgoals or steps. This technique, often facilitated by methods like Chain of Thought, enables the LLM to handle intricate problems efficiently by thinking step-by-step. It transforms big tasks into multiple manageable parts, improving performance on complex assignments.

Question 2: Why is task decomposition important for LLM agents
Answer 2: Task decomposition is crucial for LLM agents because complicated tasks involve many steps, requiring agents to plan ahead. It breaks down large, hard tasks into smaller, simpler, and more manageable subgoals. This process enables the efficient handling of complex tasks by the agent.

Question 3: Methods for task decomposition in large language model agents
Answer 3: Methods for task decomposition in large language model agents include Chain of Thought (CoT), which i

Now we feed each question & it's anwwer as a context to LLM and ask it to extract answer from this context.

In [35]:
# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = prompt | llm | StrOutputParser()

response = final_rag_chain.invoke({"context": qna_pairs, "question": QUESTION})
console.print(Markdown(response))

Task decomposition for LLM agents is the process where an agent breaks down large, complex tasks into smaller, simpler, and more manageable subgoals or steps. This technique is crucial because complicated tasks involve many steps, requiring agents to plan ahead and handle intricate problems efficiently.

By transforming big tasks into multiple manageable parts, task decomposition improves performance on complex assignments and enables the LLM to think step-by-step. This process also sheds light on the model's thinking process.

It is often facilitated by methods like Chain of Thought (CoT), which instructs the model to "think step by step," and its extension, Tree of Thoughts (ToT), which explores multiple reasoning possibilities. Additionally, the LLM can perform decomposition through simple prompting, such as "Steps for XYZ" or "What are the subgoals for achieving XYZ?", or by using task-specific instructions. Human inputs and LLM-based task planning guided by few-shot examples are a

#### Answering Recursively
In this technique, the questions list we got above is passed recursively to the LLM - first $Q_1$ is passed and we get a response $A_1$ from LLM. $Q_1$ + $A_1$ is added as a context to $Q_2$ to get $A_2$, then ($Q_1$ + $A_1$) and ($Q_2$ + $A_2$) is added as a context when passing $Q_3$ to the LLM and so on. Finally, we land up with context -> {($Q_1$ + $A_1$), ($Q_2$ + $A_2$), ..., ($Q_{N-1}$ + $A_{N-1}$) } when $Q_N$ is passed to the LLM. The answer from the LLM to QN with the above combined context is the final response. The intutition is by passing this "combined context" derived from the recursive process helps the LLM give a more coherent response to original question.

**Example**

User asks: 
* "Explain how a blockchain works and why it is secure".

Sub-questions:
* "How does a blockchain work?"
* "Why is it secure?"
* "Explain the connection between (1) and (2)."

**Process:**
* Retrieve + answer (1) 
* Pass answer (1) into answering (2) 
* Use (1) and (2) to explain (3)

**Key Properties**
* Answers accumulate → more reasoning continuity 
* Produces cohesive explanation
* Less duplication
* Less RAG calls (because previous answers seed later ones)

**Subtle difference**
* This method uses the **answer of each sub-step to inform the next**, creating a chain of reasoning — like a teacher building one idea after another.

The image below illustrates this process visually:

![Multi Query](../images/ans_recursively.png)

In [36]:
# Prompt
template = """Here is the question you need to answer:
\n --- \n {question} \n --- \n
Here is any available background question + answer pairs:
\n --- \n {q_a_pairs} \n --- \n
Here is additional context relevant to the question: 
\n --- \n {context} \n --- \n
Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [37]:
from operator import itemgetter

def format_qa_pair(question, answer):
    """Format Q and A pair"""
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

q_a_pairs = ""
for i, q in enumerate(questions):
    rag_chain = (
        {
            # get the context by asking the retriver to retrieve it
            # based on the question
            "context": itemgetter("question") | retriever,
            "question": itemgetter("question"),
            # for the first question, q_a_pairs will be ""
            "q_a_pairs": itemgetter("q_a_pairs"),
        }
        # format my prompt with above parameters
        | decomposition_prompt
        # ask LLM for response to formatted decomposition prompt
        | llm
        # parse out text as answer
        | StrOutputParser()
    )
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair
    console.print(f"[yellow]Intermediate QA-Pair #{i+1} -> [/yellow]")
    console.print(Markdown(q_a_pairs))

Intermediate QA-Pair #1 -> 
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Question: LLM agent task decomposition definition Answer: Task decomposition for an LLM agent is the process where the agent breaks down large, complex tasks into smaller, more manageable subgoals or steps. This enables the efficient handling of complex tasks by transforming them into multiple simpler, manageable parts.

Techniques like Chain of Thought (CoT) are often employed for this purpose, instructing the model to "think step by step" to utilize more computation and decompose hard tasks. Tree of Thoughts extends CoT by exploring multiple reasoning possibilities at each step, creating a tree structure of decomposed thoughts.

Task decomposition can be achieved through:

 1 LLM with simple prompting: e.g., "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?"                    
 2 Task-specific instructions: e.g., "Write a 

Notice how we keep adding a Q & A pair to the overall context. At the end of all the questions (Q&A pairs), we get the final answer from the LLM, which we will display below.

In [38]:
console.print(f"[yellow]Final answer:[/yellow]")
console.print(Markdown(answer))

Final answer:
LLM agents employ several strategies for task decomposition, breaking down complex problems into smaller, more manageable subgoals. These strategies include:

 1 Chain of Thought (CoT): This is a standard prompting technique where the LLM is explicitly instructed to "think step 
 2 Tree of Thoughts (ToT): Extending CoT, Tree of Thoughts decomposes a problem into multiple thought steps and generate
 3 LLM with Simple Prompting: The LLM can be directly prompted to decompose tasks using straightforward instructions. Ex
    • "Steps for XYZ.\n1."                                                                                              
    • "What are the subgoals for achieving XYZ?"                                                                        
    • In systems like HuggingGPT, the LLM acts as the "brain" for task planning, parsing user requests into multiple tas
 4 Task-Specific Instructions: Providing the LLM with instructions tailored to the specific task can 

## Step Back
A different approach, presented by Google, is _Step-Back Prompting_. It takes the opposoite approach, where it tries to ask a more abstract question. So [the paper](https://arxiv.org/pdf/2310.06117.pdf) talks a lot about using few-shot prompting to produce what they call the _step-back_ (or more abstract) questions. The way it does it is to provide a number of examples of step-back questions, given the original question.

**Examples**


![Step Back](../images/step_back.png)